# Autonomous Infrastructure Remediation with Structured Outputs

This notebook demonstrates an **Agentic Workflow** for Platform Engineering. We will build an autonomous Kubernetes debugging agent ("Kube-AutoFix") that detects deployment failures, reasons through the logs as a Staff-Level SRE, and uses the **OpenAI Structured Outputs** feature to guarantee a mathematically validated, auto-corrected YAML manifest.

### Why Structured Outputs for Infrastructure?
When piping LLM outputs directly into live infrastructure (like a Kubernetes cluster), hallucinated fields, missing syntax, or markdown code fences (` ```yaml `) can cause catastrophic deployment failures. 

By using `response_format`, we force the model to output exactly the JSON structure we define via Pydantic. This eliminates parsing logic, regex scraping, and retry loops just to get valid JSON—making infrastructure agents safe and production-ready.

In [ ]:
# Install required libraries
!pip install -q openai pydantic pyyaml


In [ ]:
import json
import os
from pydantic import BaseModel, Field
from openai import OpenAI

# Initialize the OpenAI client
# Ensure your OPENAI_API_KEY environment variable is set
client = OpenAI()
MODEL = "gpt-4o"


### 1. Defining the Output Schema (Structured Outputs)

We define the exact structure we want the LLM to return. The model will provide its step-by-step reasoning, identify the root cause, assign a confidence score, list the changes, and output the final corrected YAML.

In [ ]:
from pydantic import BaseModel, Field

class LLMDiagnosis(BaseModel):
    reasoning: str = Field(..., description="Step-by-step reasoning analyzing the Kubernetes failure.")
    root_cause: str = Field(..., description="A short, one-line summary of the root cause.")
    confidence_score: float = Field(..., ge=0.0, le=1.0, description="Confidence score between 0.0 and 1.0 that this fixes the issue.")
    changes_made: list[str] = Field(..., description="List of changes made to the YAML.")
    corrected_yaml: str = Field(..., description="The fully corrected Kubernetes YAML manifest.")

### 2. The Mock Kubernetes Environment

To make this notebook self-contained without requiring an active AWS EKS cluster, we will use a `MockKubernetesEnvironment`. 

This environment simulates applying a broken deployment (`nginx:latestttt`), polling its status, and generating a debug bundle containing pod descriptions and container logs when it inevitably fails with an `ImagePullBackOff`.

In [ ]:
# The initial broken YAML we want to deploy
BROKEN_YAML = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: autofix-test-nginx
  namespace: autofix-agent-env
spec:
  replicas: 2
  selector:
    matchLabels:
      app: autofix-test-nginx
  template:
    metadata:
      labels:
        app: autofix-test-nginx
    spec:
      containers:
      - name: nginx
        image: nginx:latestttt  # <-- TYPO HERE
        ports:
        - containerPort: 80
"""

import yaml

class MockKubernetesEnvironment:
    def __init__(self):
        self.deployment_yaml = ""
    
    def apply_manifest(self, yaml_str: str) -> bool:
        print("[K8s] Applying manifest...")
        self.deployment_yaml = yaml_str
        return True

    def poll_status(self) -> dict:
        print("[K8s] Polling pod status...")
        try:
            manifests = list(yaml.safe_load_all(self.deployment_yaml))
            for manifest in manifests:
                if not manifest: continue
                if manifest.get("kind") == "Deployment":
                    # 1. ENFORCE INVARIANTS (Safety Checks)
                    if manifest.get("metadata", {}).get("namespace") != "autofix-agent-env":
                        return {"status": "FAILED", "reason": "SafetyViolation: Namespace changed from autofix-agent-env"}
                    if manifest.get("spec", {}).get("replicas") != 2:
                        return {"status": "FAILED", "reason": "SafetyViolation: Replica count modified"}
                    
                    # 2. CHECK THE FIX
                    containers = manifest.get("spec", {}).get("template", {}).get("spec", {}).get("containers", [])
                    for container in containers:
                        image = container.get("image", "")
                        
                        valid_nginx_tags = ["nginx:latest", "nginx:1.27", "nginx:1.26", "nginx:alpine"]
                        
                        if image == "nginx:latestttt":
                            print("[K8s] Failure detected: ImagePullBackOff")
                            return {"status": "FAILED", "reason": "ImagePullBackOff"}
                        elif image in valid_nginx_tags:
                            print("[K8s] All pods Running and Ready!")
                            return {"status": "SUCCESS"}
                        else:
                            return {"status": "FAILED", "reason": f"ErrImagePull: {image} not found"}
            
            print("[K8s] Invalid or missing Deployment manifest")
            return {"status": "FAILED", "reason": "InvalidManifest"}
        except Exception as e:
            print(f"[K8s] YAML Parse Error: {e}")
            return {"status": "FAILED", "reason": "InvalidYAML"}

    def collect_debug_bundle(self) -> str:
        print("[K8s] Collecting pod logs and events...")
        return """\
[POD: autofix-test-nginx-54d6d7bd76-9qx22]
State: Waiting (ImagePullBackOff)
Events:
  Warning  Failed     12s (x3 over 45s)  kubelet            Failed to pull image "nginx:latestttt": rpc error: code = NotFound desc = failed to pull and unpack image "docker.io/library/nginx:latestttt": failed to resolve reference "docker.io/library/nginx:latestttt": pull access denied, repository does not exist or may require authorization: server message: insufficient_scope: authorization failed
  Warning  Failed     12s (x3 over 45s)  kubelet            Error: ErrImagePull
Logs: (none)
"""


### 3. The Staff-Level SRE System Prompt

Because this agent modifies live infrastructure, we must enforce strict operational boundaries. We instruct the model to never alter the namespace, make the minimal possible changes, and preserve all labels and resource limits.

In [ ]:
SYSTEM_PROMPT = """\
You are Kube-AutoFix, a Staff-Level Kubernetes Site Reliability Engineer (SRE).
Your job is to autonomously diagnose and repair broken Kubernetes manifests based on pod logs and events.

You MUST adhere to these strict operational boundaries:
1. NAMESPACE LOCK: All resources operate in the "autofix-agent-env" namespace. Never alter the namespace.
2. MINIMAL CHANGES: Make the SMALLEST possible change to fix the root cause. Do not refactor or "clean up" the YAML.
3. RESOURCE LIMITS: Never add or alter CPU/Memory requests or limits unless explicitly diagnosing an OOMKilled event.
4. IMAGE TAGS: If fixing a broken image tag, use a well-known stable tag (e.g., 'latest' or a major version). Never guess obscure tags.
5. NO NEW RESOURCES: Do not add Services, ConfigMaps, or Ingresses unless they are the direct cause of the failure.
6. PRESERVE STRUCTURE: Keep all existing labels, annotations, and volume mounts intact.
"""

### 4. The Autonomous Loop

Now we tie it all together. The loop will deploy the manifest, monitor it, and if it fails, send the debug bundle to GPT-4o. We use `client.beta.chat.completions.parse` with `response_format=LLMDiagnosis` to automatically parse the model's response into our Pydantic object.

In [ ]:
def run_autonomous_agent(max_iterations=3):
    env = MockKubernetesEnvironment()
    current_yaml = BROKEN_YAML
    
    for iteration in range(1, max_iterations + 1):
        print(f"\n--- Iteration {iteration} ---")
        
        # 1. Deploy
        env.apply_manifest(current_yaml)
        
        # 2. Monitor
        status = env.poll_status()
        
        if status["status"] == "SUCCESS":
            print("Autonomous remediation complete!")
            break
            
        # 3. Debug
        debug_info = env.collect_debug_bundle()
        
        # 4. Reason & Fix (Structured Outputs)
        print("[Agent] Consulting GPT-4o to diagnose the issue...")
        
        user_prompt = f"""\
I attempted to deploy the following manifest, but the pods are failing.

CURRENT MANIFEST:
{current_yaml}

DIAGNOSTIC BUNDLE:
{debug_info}

Please diagnose the issue and provide the corrected YAML.
"""
        
        response = client.beta.chat.completions.parse(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            response_format=LLMDiagnosis,
            temperature=0.2, # Low temperature for deterministic infrastructure changes
        )
        
        # The response is automatically parsed into our Pydantic schema!
        diagnosis = response.choices[0].message.parsed
        
        print("\n**LLM Diagnosis Received:**")
        print(f"- Root Cause: {diagnosis.root_cause}")
        print(f"- Confidence: {diagnosis.confidence_score * 100}%")
        print(f"- Changes Made: {diagnosis.changes_made}")
        print("\n**Reasoning:**")
        print(diagnosis.reasoning)
        
        print("\n**Corrected YAML:**")
        print(diagnosis.corrected_yaml)
        
        # --- NEW: VALIDATE YAML BEFORE REAPPLYING ---
        clean_yaml = diagnosis.corrected_yaml.strip()
        
        # Strip markdown fences if the LLM hallucinated them
        if clean_yaml.startswith("```"):
            lines = clean_yaml.split("\n")
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines[-1].startswith("```"):
                lines = lines[:-1]
            clean_yaml = "\n".join(lines)
            
        # Verify it parses as YAML
        try:
            import yaml
            yaml.safe_load(clean_yaml)
            # Prepare for the next iteration
            current_yaml = clean_yaml
        except Exception as e:
            print(f"\n[Agent] LLM generated invalid YAML: {e}")
            # Do not update current_yaml, let it retry with the same broken yaml

run_autonomous_agent()
